[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week10.ipynb)

# Week 10: Recurrent Networks (RNNs)
**Introduction to Deep Learning (HIT)** &middot; Part III: Architectures & Representation Learning

Sequence data and recurrence; the RNN cell; backpropagation through time; vanishing and exploding gradients.

**Instructor practice notebook** for the 2-hour practice lesson. Work through the sections below live, running each cell and trying the variations. The student homework is the weekly lab.

### Goals

- Build a plain RNN for sequence data.
- Understand recurrence and backpropagation through time.
- Observe the vanishing-gradient problem directly.

### Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## 1. An RNN over a sequence
It carries a hidden state forward, reusing the same weights at every step.

In [ ]:
torch.manual_seed(0)
rnn = nn.RNN(input_size=1, hidden_size=16, batch_first=True)
x = torch.randn(8, 20, 1)                 # (batch, time, features)
out, h = rnn(x)
print("output sequence:", tuple(out.shape), "| final hidden:", tuple(h.shape))
print("the same weight is used at all 20 steps:", tuple(rnn.weight_hh_l0.shape))

## 2. The recurrence BY HAND
Implement one RNN step yourself and match PyTorch's output.

In [ ]:
torch.manual_seed(0)
cell = nn.RNNCell(1, 16)
seq = torch.randn(5, 1)                    # 5 time steps, 1 feature
h = torch.zeros(16)
states = []
for t in range(5):
    h = torch.tanh(cell.weight_ih @ seq[t] + cell.bias_ih + cell.weight_hh @ h + cell.bias_hh)
    states.append(h)
# compare with the built-in cell:
h2 = torch.zeros(1, 16)
for t in range(5):
    h2 = cell(seq[t:t+1], h2)
print("manual recurrence matches RNNCell:", torch.allclose(states[-1], h2.squeeze(), atol=1e-5))

## 3. Vanishing gradients, measured
Gradient of the LAST output w.r.t. each input step shrinks for early steps.

In [ ]:
T = 40
rnn = nn.RNN(1, 8, batch_first=True)
x = torch.randn(1, T, 1, requires_grad=True)
out, _ = rnn(x)
out[:, -1].sum().backward()
g = x.grad.abs().squeeze().tolist()
plt.plot(range(T), g); plt.xlabel("time step"); plt.ylabel("|grad of last output|")
plt.title("Early steps receive tiny gradients"); plt.show()
print("grad at step 0:", f"{g[0]:.2e}", "| grad at last step:", f"{g[-1]:.2e}")

## 4. Exploding gradients and clipping
Large recurrent weights make the gradient blow up; clipping caps its norm.

In [ ]:
rnn = nn.RNN(1, 8, batch_first=True)
with torch.no_grad():
    rnn.weight_hh_l0 *= 3.0                 # force instability
x = torch.randn(1, 50, 1, requires_grad=True)
rnn(x)[0][:, -1].sum().backward()
raw = x.grad.norm().item()
o = torch.optim.SGD(rnn.parameters(), 0.1)
clipped = torch.nn.utils.clip_grad_norm_(rnn.parameters(), max_norm=1.0)
print(f"input-grad norm (huge): {raw:.1f} | param-grad norm before clip: {clipped:.1f} -> capped at 1.0")

## 5. A tiny sequence task end to end
Predict whether a short random walk ends positive.

In [ ]:
torch.manual_seed(0)
seqs = torch.randn(400, 15, 1)
labels = (seqs.sum(dim=1).squeeze() > 0).long()
class SeqClf(nn.Module):
    def __init__(self):
        super().__init__(); self.rnn = nn.RNN(1, 16, batch_first=True); self.head = nn.Linear(16, 2)
    def forward(self, x):
        out, _ = self.rnn(x); return self.head(out[:, -1])     # last hidden -> 2 classes
m = SeqClf(); o = torch.optim.Adam(m.parameters(), 0.01); f = nn.CrossEntropyLoss()
for _ in range(120):
    o.zero_grad(); f(m(seqs), labels).backward(); o.step()
print("accuracy:", round((m(seqs).argmax(1) == labels).float().mean().item(), 3))

## 6. Hidden size is a capacity knob
More hidden units = more memory, more parameters.

In [ ]:
for hs in [2, 8, 32]:
    torch.manual_seed(0)
    m = nn.RNN(1, hs, batch_first=True)
    print(f"hidden_size={hs:2d}: params {sum(p.numel() for p in m.parameters()):4d}")

## 7. Many-to-many: an output at every step
Use the full output sequence, not just the last hidden state.

In [ ]:
rnn = nn.RNN(1, 8, batch_first=True); head = nn.Linear(8, 1)
x = torch.randn(4, 10, 1)
out, _ = rnn(x)
per_step = head(out)                       # (batch, time, 1): a prediction per time step
print("per-step output shape:", tuple(per_step.shape))

**Try it live:** raise the sequence length in section 5 to 60 and watch accuracy drop, the vanishing-gradient problem in action.

### Mini-exercise
Train the section-5 classifier at sequence lengths 15 and 60 and report both accuracies. Why does the longer sequence hurt a plain RNN?

*Try it before revealing the solution below.*

**Solution.**

In [ ]:
def run(T):
    torch.manual_seed(0)
    s = torch.randn(400, T, 1); lab = (s.sum(dim=1).squeeze() > 0).long()
    m = SeqClf(); o = torch.optim.Adam(m.parameters(), 0.01); f = nn.CrossEntropyLoss()
    for _ in range(120):
        o.zero_grad(); f(m(s), lab).backward(); o.step()
    return (m(s).argmax(1) == lab).float().mean().item()
print("T=15 accuracy:", round(run(15), 3))
print("T=60 accuracy:", round(run(60), 3), "(gradients from early steps vanish)")

**Recap:** RNNs share weights across time; unrolling makes them deep; long-range gradients vanish or explode; clip to tame explosion, and use gating (next week) to fight vanishing.

---
Student materials for this week: the lab handout (`labs/week10.html`) and the curated references (`references/week10.html`) in the course site.